# Qwen3-VL text-to-image retrieval

Embed a small image gallery once, score a batch of text queries against it, and plot the similarity heatmap + top-K images per query.

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import mlx.core as mx
import numpy as np
import requests
from PIL import Image

from mlx_embeddings import load

GALLERY = [
    ("woman with dog on beach",
     "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"),
    ("two cats on a couch",
     "http://images.cocodataset.org/val2017/000000039769.jpg"),
    ("tennis player on a court",
     "http://images.cocodataset.org/val2017/000000000872.jpg"),
    ("bear in the wild",
     "http://images.cocodataset.org/val2017/000000000285.jpg"),
    ("dark train tunnel",
     "http://images.cocodataset.org/val2017/000000001268.jpg"),
    ("group of people standing together",
     "http://images.cocodataset.org/val2017/000000001000.jpg"),
]
QUERIES = [
    "a person spending time with their pet outdoors at sunset",
    "sleepy cats relaxing indoors",
    "someone playing a racquet sport",
    "wildlife in a natural habitat",
    "the inside of a transit tunnel",
    "a crowd of people gathered outside",
]
INSTRUCTION = "Retrieve images that match the user's query."
TOP_K = 3

def fetch(src):
    if src.startswith(("http://", "https://")):
        return Image.open(BytesIO(requests.get(src, timeout=30).content)).convert("RGB")
    return Image.open(src).convert("RGB")

labels, urls = zip(*GALLERY)
images = [fetch(u) for u in urls]

In [ ]:
model, processor = load("Qwen/Qwen3-VL-Embedding-2B")

img_embeds = model.process([{"image": i} for i in images], processor=processor)
txt_embeds = model.process(
    [{"text": q, "instruction": INSTRUCTION} for q in QUERIES], processor=processor,
)
sim = np.array((txt_embeds @ img_embeds.T).astype(mx.float32))

for qi, q in enumerate(QUERIES):
    top = np.argsort(-sim[qi])[:TOP_K]
    print(f"q{qi}: {q}")
    for k, idx in enumerate(top):
        print(f"  #{k + 1}  {sim[qi, idx]:.3f}  {labels[idx]}")

In [ ]:
fig, (ax_hm, ax_tk) = plt.subplots(
    2, 1, figsize=(2 + TOP_K * 3, len(QUERIES) * 2.2),
    gridspec_kw={"height_ratios": [1, 2]},
)

# Heatmap
im = ax_hm.imshow(sim, cmap="viridis", aspect="auto")
ax_hm.set_xticks(range(len(GALLERY)), labels, rotation=30, ha="right", fontsize=8)
ax_hm.set_yticks(range(len(QUERIES)), [f"q{i}" for i in range(len(QUERIES))], fontsize=8)
ax_hm.set_title("text \u2194 image similarity")
thresh = sim.mean()
for i in range(sim.shape[0]):
    for j in range(sim.shape[1]):
        ax_hm.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=7,
                   color="white" if sim[i, j] < thresh else "black")
fig.colorbar(im, ax=ax_hm, shrink=0.8)

# Top-K images per query
ax_tk.axis("off")
gs = ax_tk.get_subplotspec().subgridspec(len(QUERIES), TOP_K + 1, wspace=0.1, hspace=0.4)
for qi, q in enumerate(QUERIES):
    lbl_ax = fig.add_subplot(gs[qi, 0])
    lbl_ax.axis("off")
    lbl_ax.text(1.0, 0.5, f"q{qi}: {q}", ha="right", va="center", fontsize=9, wrap=True)
    for k, idx in enumerate(np.argsort(-sim[qi])[:TOP_K]):
        ax = fig.add_subplot(gs[qi, k + 1])
        ax.imshow(images[idx])
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"#{k + 1}  {sim[qi, idx]:.2f}\n{labels[idx]}", fontsize=8,
                     color="tab:green" if k == 0 else "black")

fig.suptitle("Qwen3-VL retrieval", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()